# Ejecución local

Hasta ahora, todos los circuitos se han ejecutado con `infrastructure="local"`. Este notebook explica qué hace ese backend, cómo repartir shots entre varias ejecuciones con `n_qpus`, y cómo fijar la aleatoriedad con `seed`. [`03b_escalar_cunqa_qmio.ipynb`](03b_escalar_cunqa_qmio.ipynb) cubre las infraestructuras que necesitan un clúster: CUNQA y QMIO.

## Qué hay detrás de "local"

`infrastructure="local"` ejecuta por defecto el circuito con el simulador Aer de Qiskit, en el propio ordenador. `result.backend` lo confirma:

In [1]:
import polypus

bell = polypus.Circuit(2).h(0).cx(0, 1).measure_all()
result = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local")
print(result.backend)

aer


## El backend nativo

Además de `"aer"`, Polypus tiene un motor de simulación propio en Rust, seleccionable con el argumento `backend`:

In [2]:
result_nativo = polypus.run_quantum_circuit(
    bell, shots=1000, infrastructure="local", backend="polypus"
)
print(result_nativo.backend, result_nativo.counts)

polypus [{'00': 518, '11': 482}]


El resultado es el mismo estado de Bell, solo cambia qué motor lo calcula. El backend nativo también admite `fusion`, que combina puertas consecutivas antes de simular para ir más rápido: se activa por defecto, y `fusion=False` fuerza una ejecución puerta a puerta. Pasar `fusion=True` en `"aer"`, que no sabe fusionar, da un error en vez de ignorarlo en silencio:

In [3]:
try:
    polypus.run_quantum_circuit(
        bell, shots=1000, infrastructure="local", backend="aer", fusion=True
    )
except ValueError as e:
    print(e)

fusion=True applies only to the native backend="polypus"; backend "aer" on infrastructure "local" cannot fuse gates. Omit fusion, or pass fusion=False for a gate-by-gate run.


Las diferencias de rendimiento entre ambos backends están documentadas, con benchmarks reproducibles, en el README principal.

## Repartir shots con `n_qpus`

`n_qpus` reparte la ejecución de los shots entre varias instancias en paralelo, para reducir el tiempo de reloj. Con `n_qpus=1`, el valor por defecto, el resultado viene envuelto en una lista de un único elemento, como se vio en [`01_primer_circuito.ipynb`](01_primer_circuito.ipynb):

In [4]:
r1 = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local", n_qpus=1)
print(r1.counts)

[{'00': 489, '11': 511}]


Con `n_qpus` mayor que 1, `result.counts` mantiene la misma forma de lista, ahora con un diccionario por réplica: cada réplica contiene solo los shots que le tocaron, no el total completo.

In [5]:
r4 = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local", n_qpus=4)
print(len(r4.counts), r4.counts)

4 [{'00': 128, '11': 122}, {'00': 129, '11': 121}, {'00': 117, '11': 133}, {'00': 139, '11': 111}]


Esto es un motivo real de error: código que lee solo `result.counts[0]` esperando la ejecución completa obtiene en cambio solo la parte de la primera réplica, descartando en silencio el resto de los shots. `result.merged_counts` es el arreglo: la suma clave a clave de todas las réplicas, como un único diccionario, el mismo total que produjo toda la ejecución sin importar cuántas réplicas use:

In [6]:
print(r4.merged_counts)
print(sum(r4.merged_counts.values()) == 1000)

{'00': 513, '11': 487}
True


`result.merged_counts` coincide con `result.counts[0]` cuando `n_qpus` es 1, así que leerlo por defecto es seguro siempre que lo que importe sea el total sobre todos los shots, sin importar cuántas réplicas haya usado la ejecución.

## Reproducibilidad con `seed`

Por defecto, cada ejecución usa una semilla aleatoria distinta, tomada del sistema operativo. Pasar `seed` fija esa aleatoriedad, y el resultado es idéntico entre ejecuciones:

In [7]:
a = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local", seed=42)
b = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local", seed=42)
print(a.counts == b.counts)

True


Sin pasar `seed`, `result.seed` informa de la semilla que se usó, para poder reproducir esa misma ejecución más tarde:

In [8]:
sin_semilla = polypus.run_quantum_circuit(bell, shots=1000, infrastructure="local")
print(sin_semilla.seed)

11688252215994569869


## Otros campos del resultado

Además de `counts`, `seed` y `backend`, `result.id` guarda un identificador único de la ejecución, y `result.infrastructure` repite el valor de `infrastructure` usado en la llamada. Son útiles para registrar y encontrar una ejecución concreta más adelante.

## Resumen

Este notebook ha explicado qué hace `infrastructure="local"`, la elección entre el backend `"aer"` y el nativo `"polypus"` con `fusion`, cómo `n_qpus` cambia el reparto de shots mientras `result.counts` mantiene un diccionario por réplica, por qué leer solo `result.counts[0]` descarta shots en silencio, `result.merged_counts` como el total seguro, y cómo `seed` da reproducibilidad.

La siguiente sección trata las infraestructuras que necesitan un clúster: CUNQA y QMIO.

## Siguiente paso

Continúa con: [`03b_escalar_cunqa_qmio.ipynb`](03b_escalar_cunqa_qmio.ipynb).